# Fine-tune local / Colab

## Mục tiêu
Epoch là một lượt qua train set; batch là số ảnh mỗi bước; learning rate điều khiển mức cập nhật. AMP giảm bộ nhớ. Smoke 1 epoch kiểm pipeline, không chứng minh cải thiện. Không train khi đang phục vụ kiosk.

## Setup
Chạy từ repo đã clone ở revision bạn ghi nhận. Local dùng venv API; Colab xem docs/RESEARCH.md. Không tự cài dependency hoặc tải dữ liệu khi Run all.

In [1]:
from pathlib import Path
import os, sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai/research").is_dir()), None)
assert ROOT is not None, "Clone repo và chạy từ repo/notebooks"
sys.path.insert(0, str(ROOT))
os.environ["YOLO_AUTOINSTALL"] = "false"
from ai.research.cli import environment
print(json.dumps(environment(), indent=2))

{
  "python": "3.11.9",
  "platform": "Windows-10-10.0.26200-SP0",
  "versions": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "ultralytics": "8.4.162",
    "numpy": "1.26.4",
    "pillow": "10.3.0"
  },
  "cuda_available": true,
  "gpu": "NVIDIA GeForce RTX 3060"
}


## Các bước
Đọc cấu hình trước khi thực thi; các thao tác tốn tài nguyên mặc định tắt.

In [2]:
from ai.research.experiments import train, resume_checkpoint
DATASET = None
CHECKPOINT = None  # đường dẫn local tới yolov8s-pose.pt đã duyệt
EXECUTE = False
if DATASET is None or CHECKPOINT is None:
    print("NOT EXECUTED: cần dataset release và checkpoint tin cậy.")
else:
    plan = train(Path(CHECKPOINT), Path(DATASET), ROOT / "ai/artifacts/smoke-r1",
                 epochs=1, batch=2, imgsz=640, workers=0, execute=EXECUTE)
    print(plan)
# Resume chỉ dùng cho run bị ngắt, giữ dataset/config gốc:
# resume_checkpoint(Path(".../training/weights/last.pt"))

NOT EXECUTED: cần dataset release và checkpoint tin cậy.


## Kiểm tra
Output ghi rõ thao tác thực sự chạy và thao tác bị bỏ qua. Thiếu dữ liệu không được thay bằng số giả. Khi thay dữ liệu/cấu hình, restart kernel và Run all.

## Bước tiếp theo
Colab: review code trước mount Drive; copy archive vào runtime, kiểm checksum, extract bằng CLI. GPU/quota thay đổi. Backup checkpoint + metadata sang nơi bền vững do bạn chọn; không tự upload ảnh khách. Dataset mới = run mới.